In [ ]:
%load_ext autoreload
%autoreload 2

import os

import matplotlib.pyplot as plt
import pandas as pd
from hydra import compose, initialize
from hydra.core.global_hydra import GlobalHydra  # Import GlobalHydra explicitly
from hydra.utils import instantiate

from ogbench.utils.config_resolvers import (
    get_default_transform,
    get_monitor_metric,
    get_monitor_mode,
    infer_in_channels,
)

# Clear GlobalHydra instance if already initialized
if GlobalHydra().is_initialized():
    GlobalHydra().clear()

initialize(config_path="../configs", job_name="job")

In [ ]:
dataset_name = "addneuromed"

cfg = compose(
    config_name="train.yaml",
    overrides=[
        "model=gat",
        f"dataset={dataset_name}",
        "dataset.loader.parameters.adjacency_threshold=0.5",
        "dataset.loader.parameters.node_sample_ratio=full",
    ],
    return_hydra_config=True,
)
loader = instantiate(cfg.dataset.loader)
dataset = loader.load_dataset()
print(dataset.processed_dir)
print(dataset[0])


def load_dataset(dataset_name, adj_thresh=0.5):
    """
    Load the FTD dataset with a specified adjacency threshold.
    """
    cfg = compose(
        config_name="train.yaml",
        overrides=[
            "model=gat",
            f"dataset={dataset_name}",
            f"dataset.loader.parameters.adjacency_threshold={adj_thresh}",
            "dataset.loader.parameters.node_sample_ratio=full",
        ],
        return_hydra_config=True,
    )
    loader = instantiate(cfg.dataset.loader)
    dataset = loader.load_dataset()
    return dataset

In [ ]:
root = "/home/anon/code/ogbench/run_data/omics/"
name = osp.join(
    root,
    f"{dataset.data_name}",
    f"adj_thresh_{dataset.adjacency_threshold}",
    f"{dataset.method}",
    f"p_{dataset.node_sample_ratio}",
    f"train_split_{dataset.train_val_test_split[0]}",
    "raw/adj_matrix.npy",
)
print(name)
adj_matrix = np.load(name)

In [ ]:
# Adjacency matrix loaded

In [ ]:
def get_graph_stats(dataset):
    """
    Get statistics of the graph.
    """
    # Load the adjacency matrix
    root = "/home/anon/code/ogbench/run_data/omics/"
    name = osp.join(
        root,
        f"{dataset.data_name}",
        f"adj_thresh_{dataset.adjacency_threshold}",
        f"{dataset.method}",
        f"p_{dataset.node_sample_ratio}",
        f"train_split_{dataset.train_val_test_split[0]}",
        "raw/adj_matrix.npy",
    )
    adj_matrix = np.load(name)

    # Generate a graph from the adjacency matrix
    graph = nx.from_numpy_array(adj_matrix)
    graph.remove_edges_from(nx.selfloop_edges(graph))

    # Calculate statistics
    num_nodes = graph.number_of_nodes()
    num_edges = graph.number_of_edges()
    avg_degree = np.mean([d for _, d in graph.degree()])
    density = nx.density(graph)
    number_connected_components = nx.number_connected_components(graph)

    return {
        "num_nodes": num_nodes,
        "num_edges": num_edges,
        "avg_degree": avg_degree,
        "density": density,
        "number_connected_components": number_connected_components,
    }


# Get graph statistics
stats = get_graph_stats(dataset)
print("\nGraph statistics:\n")
for key, value in stats.items():
    print(f"\t{key}: {value}")

In [ ]:
# Initialize a list to store all stats
all_stats = []

for adj_thresh in np.arange(0, 1.01, 0.01):
    # Load the dataset
    dataset = load_dataset(dataset_name, adj_thresh=adj_thresh)
    # Get graph statistics
    stats = get_graph_stats(dataset)
    # Add the adjacency threshold to the stats
    stats["adj_thresh"] = adj_thresh
    # Append the stats to the list
    all_stats.append(stats)

# Save all stats to a CSV file
output_file = f"./stats/{dataset_name}/graph_stats.csv"
os.makedirs(os.path.dirname(output_file), exist_ok=True)
with open(output_file, "w", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "adj_thresh",
            "num_nodes",
            "num_edges",
            "avg_degree",
            "density",
            "number_connected_components",
        ],
    )
    writer.writeheader()  # Write the header row
    writer.writerows(all_stats)  # Write all rows

print(f"Graph statistics saved to {output_file}")

In [ ]:
# import os, csv, math
# import numpy as np
# from concurrent.futures import ProcessPoolExecutor, as_completed
# from functools import partial
# from time import perf_counter

# # ---------- helper: single-task worker ----------
# def _compute_stats_for_thresh(adj_thresh, dataset_name):
#     """
#     Runs in a separate process. Must be top-level so it can be pickled.
#     """
#     # Load the dataset for this threshold
#     dataset = load_dataset(dataset_name, adj_thresh=adj_thresh)
#     # Compute graph stats
#     stats = get_graph_stats(dataset)
#     # Attach the threshold
#     stats["adj_thresh"] = float(adj_thresh)
#     return stats

# # ---------- main parallel block ----------
# start = perf_counter()

# # Make a stable list of thresholds (avoids float step accumulation issues)
# adj_thresholds = [round(x, 2) for x in np.linspace(0.0, 1.0, 101)]

# # How many worker processes to use. Tweak if you want fewer.
# max_workers = os.cpu_count() or 2

# # Run in parallel
# results_in_order = [None] * len(adj_thresholds)
# with ProcessPoolExecutor(max_workers=max_workers) as ex:
#     # Submit all jobs
#     futures = {
#         ex.submit(_compute_stats_for_thresh, t, dataset_name): idx
#         for idx, t in enumerate(adj_thresholds)
#     }
#     # Collect as they finish, but store back in original order
#     for fut in as_completed(futures):
#         idx = futures[fut]
#         try:
#             results_in_order[idx] = fut.result()
#         except Exception as e:
#             # You can choose to raise here, or record an error row.
#             # For now, record a minimal row with the error noted.
#             results_in_order[idx] = {
#                 "adj_thresh": adj_thresholds[idx],
#                 "num_nodes": None,
#                 "num_edges": None,
#                 "avg_degree": None,
#                 "density": None,
#                 "number_connected_components": None,
#                 "error": str(e),
#             }

# # Save to CSV (same columns you used; "error" is optional)
# output_file = f"./stats/{dataset_name}/graph_stats.csv"
# os.makedirs(os.path.dirname(output_file), exist_ok=True)

# fieldnames = [
#     "adj_thresh",
#     "num_nodes",
#     "num_edges",
#     "avg_degree",
#     "density",
#     "number_connected_components",
# ]
# # If any row had an error, include that column so you can see what failed.
# if any(("error" in r) for r in results_in_order):
#     fieldnames.append("error")

# with open(output_file, "w", newline="") as f:
#     writer = csv.DictWriter(f, fieldnames=fieldnames)
#     writer.writeheader()
#     writer.writerows(results_in_order)

# print(f"Graph statistics saved to {output_file}")
# print(f"Completed in {perf_counter() - start:.2f}s using {max_workers} workers.")

In [ ]:
for i in range(100, -1, -1):
    print(i / 100)

In [ ]:
for datasets in ["addneuromed", "parkinsons", "covidaki", "motrpac"]:
    csv_file = f"./stats/{datasets}/graph_stats.csv"
    df = pd.read_csv(csv_file)
    # Sort the DataFrame by the 'adj_thresh' column in ascending order
    df = df.sort_values(by="adj_thresh", ascending=True)

    # Plot the evolution of graph statistics with respect to adj_thresh
    plt.figure(figsize=(14, 10))

    # Plot number of edges
    plt.subplot(3, 2, 1)
    plt.plot(df["adj_thresh"], df["num_edges"], label="Number of Edges", color="green")
    plt.xlabel("Adjacency Threshold")
    plt.ylabel("Number of Edges")
    plt.title("Number of Edges vs. Adjacency Threshold")
    plt.grid(True)

    # Plot average degree
    plt.subplot(3, 2, 2)
    plt.plot(df["adj_thresh"], df["avg_degree"], label="Average Degree", color="orange")
    plt.xlabel("Adjacency Threshold")
    plt.ylabel("Average Degree")
    plt.title("Average Degree vs. Adjacency Threshold")
    plt.grid(True)

    # Plot density
    plt.subplot(3, 2, 3)
    plt.plot(df["adj_thresh"], df["density"], label="Density", color="red")
    plt.xlabel("Adjacency Threshold")
    plt.ylabel("Density")
    plt.title("Density vs. Adjacency Threshold")
    plt.grid(True)

    # Plot number of connected components
    plt.subplot(3, 2, 4)
    plt.plot(
        df["adj_thresh"],
        df["number_connected_components"],
        label="Connected Components",
        color="purple",
    )
    plt.xlabel("Adjacency Threshold")
    plt.ylabel("Number of Connected Components")
    plt.title("Connected Components vs. Adjacency Threshold")
    plt.grid(True)

    plt.suptitle(datasets, fontsize=16, y=1.02)
    # Adjust layout and show the plots
    plt.tight_layout()
    plt.show()

In [ ]:
metrics = ["wgcna", "spearman_correlation", "mutual_information", "distance_correlation"]

for metric in metrics:
    # Load the CSV file into a pandas DataFrame
    csv_file = "./stats/" + metric + "/graph_stats.csv"
    df = pd.read_csv(csv_file)
    # Sort the DataFrame by the 'adj_thresh' column in ascending order
    df = df.sort_values(by="adj_thresh", ascending=True)

    # Plot the evolution of graph statistics with respect to adj_thresh
    plt.figure(figsize=(14, 10))
    plt.suptitle(metric, fontsize=18)

    # Plot number of edges
    plt.subplot(3, 2, 1)
    plt.plot(df["adj_thresh"], df["num_edges"], label="Number of Edges", color="green")
    plt.xlabel("Adjacency Threshold")
    plt.ylabel("Number of Edges")
    plt.title("Number of Edges vs. Adjacency Threshold")
    plt.yscale("log")  # Set y-axis to logarithmic scale
    plt.grid(True)

    # Plot average degree
    plt.subplot(3, 2, 2)
    plt.plot(df["adj_thresh"], df["avg_degree"], label="Average Degree", color="orange")
    plt.xlabel("Adjacency Threshold")
    plt.ylabel("Average Degree")
    plt.title("Average Degree vs. Adjacency Threshold")
    plt.yscale("log")  # Set y-axis to logarithmic scale
    plt.grid(True)

    # Plot density
    plt.subplot(3, 2, 3)
    plt.plot(df["adj_thresh"], df["density"], label="Density", color="red")
    plt.xlabel("Adjacency Threshold")
    plt.ylabel("Density")
    plt.title("Density vs. Adjacency Threshold")
    plt.yscale("log")  # Set y-axis to logarithmic scale
    plt.grid(True)

    # Plot number of connected components
    plt.subplot(3, 2, 4)
    plt.plot(
        df["adj_thresh"],
        df["number_connected_components"],
        label="Connected Components",
        color="purple",
    )
    plt.xlabel("Adjacency Threshold")
    plt.ylabel("Number of Connected Components")
    plt.title("Connected Components vs. Adjacency Threshold")
    plt.yscale("log")  # Set y-axis to logarithmic scale
    plt.grid(True)

    # Adjust layout and show the plots
    plt.tight_layout()
    plt.show()
    plt.close()